In [1]:
import os, glob, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 60)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.25})

V4 = os.path.abspath("../../../wildfire-datasets-v4")     # the dataset folder (sits next to wildfire-ml-v4)
DS = V4 + "/dataset"                                       # built datasets
RAW = V4 + "/inputs_raw"                                   # raw downloads, exactly as they arrived
START = pd.Timestamp("2019-01-01")                         # day 0 of the cube; day t = START + t days

# 00. Overview: all our data, and the road from raw files to model input

Short version of the whole project:

> *Each day, for each forest cell of about 11 km x 11 km, will a satellite see a fire there tomorrow?*

To answer this we collect **raw data** from different places, turn it into **one number table per cell per day**
(the *cube*), attach a **yes/no label** to every row, and then cut the cube into small pieces the **models** can read.

This notebook is the map. The next notebooks open each data source one by one. Read them in order:

| Notebook | What it shows |
|---|---|
| 01 | Satellite fire detections (FIRMS): what the label is made of |
| 02 | Forest boundaries and the grid of cells |
| 03 | Weather (ERA5-Land): raw file, then the 11 weather channels |
| 04 | Plants and land cover (NDVI, WorldCover) |
| 05 | Terrain, people, roads |
| 06 | Labels and rows: what is a "sample", and the train / validation / test split |
| 07 | From the cube to what the models read |
| 08 | Weather forecast inputs (GFS), the new addition |

## The six steps

1. **Raw** files, as downloaded (CSV, GeoTIFF, GeoParquet, OSM). Never edited.
2. **Cells.** The map is cut into squares of 0.1 degree. Only cells with enough forest land are kept.
3. **Daily features.** For every kept cell and every day from 2019-01-01 to 2025-12-31, we fill in 33 numbers (weather, plants, land, history of fires...).
4. **Labels.** For every cell and day we also write down: did the satellite see a fire in this cell the **next** day?
5. **Cube.** All features are stored in one big array: `[days, cells, channels]`.
6. **Model input.** For one sample (a cell and a day) we cut out the last 10 days around the cell and scale the numbers.

## Where each raw source lives and how big it is

In [2]:
rows = []
for name, what in [("firms_main_south", "fire detections, main states + south (CSV, 5-day files)"),
                   ("firms_hpn", "fire detections, Himachal / north (CSV)"),
                   ("firms_ne", "fire detections, north-east (CSV)"),
                   ("gee_new_states", "weather, NDVI, elevation, land cover (CSV exported from Google Earth Engine)"),
                   ("worldpop", "population grid (GeoTIFF, 1 km)"),
                   ("osm", "roads (OpenStreetMap extract, .pbf)"),
                   ("gfs_rda", "weather forecast (GFS, NCAR RDA), processed monthly files")]:
    files = [os.path.join(d, f) for d, _, fs in os.walk(f"{RAW}/{name}") for f in fs]
    rows.append((name, what, len(files), round(sum(os.path.getsize(f) for f in files) / 1e9, 2)))
for name, what in [("fsi_boundaries", "forest boundaries (GeoParquet)")]:
    files = [os.path.join(d, f) for d, _, fs in os.walk(f"{DS}/{name}") for f in fs]
    rows.append((name + " (in dataset/)", what, len(files), round(sum(os.path.getsize(f) for f in files) / 1e9, 2)))
pd.DataFrame(rows, columns=["folder", "what it is", "files", "size GB"])

,folder,what it is,files,size GB
0,firms_main_south,"fire detections, main states + south (CSV, 5-day files)",2048,0.65
1,firms_hpn,"fire detections, Himachal / north (CSV)",1024,0.00
2,firms_ne,"fire detections, north-east (CSV)",1460,0.27
3,gee_new_states,"weather, NDVI, elevation, land cover (CSV exported from ...",110,4.09
4,worldpop,"population grid (GeoTIFF, 1 km)",2,0.04
5,osm,"roads (OpenStreetMap extract, .pbf)",1,0.84
6,gfs_rda,"weather forecast (GFS, NCAR RDA), processed monthly files",250,0.41
7,fsi_boundaries (in dataset/),forest boundaries (GeoParquet),5,0.70


## The sets

We built the data in groups of states ("sets"), then **merged them into one training table** (`forest_all`)
and kept the three **blind states** (Uttar Pradesh, West Bengal, Kerala) in a separate table (`forest_unseen`).
Blind means: the model never sees them while we develop. They are scored only once, at the very end.

In [3]:
cells = pd.read_parquet(f"{DS}/forest_all/cells.parquet")
rows = pd.read_parquet(f"{DS}/forest_all/rows.parquet", columns=["set", "split_t", "valid_primary", "y_forest"])
ucells = pd.read_parquet(f"{DS}/forest_unseen/cells.parquet")
urows = pd.read_parquet(f"{DS}/forest_unseen/rows.parquet", columns=["set", "split_t", "valid_primary", "y_forest"])

t = cells.groupby("set").agg(cells=("cell_id", "size"), states=("state", lambda s: ", ".join(sorted(s.unique()))))
t["rows"] = rows.groupby("set").size()
t["rows used (valid_primary)"] = rows[rows.valid_primary].groupby("set").size()
t["fire rows"] = rows[rows.valid_primary].groupby("set").y_forest.sum()
t["fire %"] = (100 * t["fire rows"] / t["rows used (valid_primary)"]).round(2)
print("merged training table: forest_all")
display(t)
print("blind table: forest_unseen ->", len(ucells), "cells,", len(urows), "rows, states:", sorted(ucells.state.unique()))

merged training table: forest_all


,cells,states,rows,rows used (valid_primary),fire rows,fire %
set,,,,,,
D1,518,AP,1324526,1291333,42148,3.26
D2,310,UK,792670,770604,24011,3.12
himachal,239,Himachal Pradesh,611123,598305,10441,1.75
northeast,766,"Arunachal Pradesh, Assam, Manipur, Mizoram, Sikkim, Tripura",1958662,1916527,41585,2.17
six,4588,"CG, JH, MH, MP, OD, TS",11731516,11413301,316867,2.78
south,727,"Karnataka, Tamil Nadu",1858939,1814633,23306,1.28


blind table: forest_unseen -> 526 cells, 1344982 rows, states: ['Kerala', 'Uttar Pradesh', 'West Bengal']


## The 33 channels of the cube

Every row of the model input has these 33 numbers. They come from the sources above:

In [4]:
names = [str(n) for n in np.load(f"{DS}/scaler_forest_A_train.npz", allow_pickle=True)["names"]]
group = {}
for n in names:
    if n.startswith(("t2m", "rh_", "vpd", "wind", "precip", "days_since_rain", "soil")): g = "weather (ERA5-Land)"
    elif n.startswith("ndvi"): g = "plants (NDVI, MODIS)"
    elif n.endswith("_f") or n.startswith(("fire_days", "days_since_last_fire", "has_history")): g = "fire history (FIRMS)"
    elif n.startswith("doy"): g = "calendar"
    elif n in ("forest_fraction", "shrub_fraction", "grass_fraction", "crop_fraction"): g = "land cover (WorldCover)"
    elif n in ("elevation", "slope"): g = "terrain (Copernicus DEM)"
    elif n in ("population_density", "distance_to_roads_km"): g = "people (WorldPop, OSM roads)"
    else: g = "forest boundary + neighbours"
    group[n] = g
table = pd.DataFrame({"channel": names, "group": [group[n] for n in names]})
table.index.name = "index"
display(table.groupby("group").channel.apply(lambda s: ", ".join(s)).to_frame())
print(len(names), "channels in total")

,channel
group,
calendar,"doy_sin, doy_cos"
fire history (FIRMS),"fire_days_7d, fire_days_30d, days_since_last_fire, has_h..."
forest boundary + neighbours,"forest_land_frac, nonforest_land_ring8, crop_ring8"
land cover (WorldCover),"forest_fraction, shrub_fraction, grass_fraction, crop_fr..."
"people (WorldPop, OSM roads)","population_density, distance_to_roads_km"
"plants (NDVI, MODIS)","ndvi, ndvi_age_days"
terrain (Copernicus DEM),"elevation, slope"
weather (ERA5-Land),"t2m_max, t2m_mean, rh_min_proxy, vpd_max_proxy, wind_spe..."


33 channels in total


**Two kinds of fire history:** the first set (`fire_days_7d`, ...) counts any detection in the cell. The second set (ending `_f`) counts only
detections whose footprint is inside the forest boundary. The boundary-based one is what we use for the label.

Plus **8 forecast channels (GFS)** that we are adding. See notebook 08.